In [9]:
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

llm = ChatOllama(model="llama3.2:1b")

class AgentState(TypedDict):
    messages: List


def personal_assistant(state: AgentState):
    system = SystemMessage(content="You are a helpful personal assistant. Help the user with their tasks.")
    response = llm.invoke([system] + state["messages"])
    return {"messages": state["messages"] + [response]}


memory = MemorySaver()

graph = StateGraph(AgentState)
graph.add_node("assistant", personal_assistant)
graph.set_entry_point("assistant")
graph.add_edge("assistant", END)

app = graph.compile(checkpointer=memory)
    

In [ ]:
config = {"configurable": {"thread_id": "user-session-1"}}

result = app.invoke(
    {"messages": [HumanMessage(content="My favorite color is #46778F, a deep teal blue.")]},
    config=config
)

print(result["messages"][-1].content)


In [ ]:
result = app.invoke(
    {"messages": [HumanMessage(content="What is my favorite color?")]},
    config=config 
)
print(result["messages"][-1].content)


In [21]:
state_snapshot = app.get_state(config)
print(state_snapshot)

StateSnapshot(values={'messages': [HumanMessage(content='What is my favorite color?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'd love to help you with that! However, I'm a large language model, I don't have any information about your personal preferences or experiences. But I can suggest some fun ways for you to find out what your favorite color is!\n\nYou could try looking at a color palette or swatch book in your home, and seeing which colors stand out to you the most. Or, you could ask a friend or family member who knows you well what their favorite color is.\n\nAlternatively, you could take a quick online quiz, such as this one: What's My Favorite Color? It'll ask you a series of questions about your preferences and personality traits, and then suggest some colors based on your results.\n\nWhich option sounds most appealing to you?", additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-05-20T01:32:45.313086Z', 'done': True, '

In [ ]:
from langchain_classic.memory import ConversationEntityMemory

entity_memory = ConversationEntityMemory(llm=ChatOllama(model="llama3.2:1b"))

# Simulate storing entity facts
entity_memory.save_context(
    {"input": "My favorite color is #46778F, a deep teal blue."},
    {"output": "Got it, I'll remember that your favorite color is #46778F."}
)

# Retrieve entity memory
entity_results = entity_memory.load_memory_variables({"input": "What is my favorite color?"})
print(entity_results)

In [ ]:
from langchain_classic.memory import ConversationEntityMemory
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List, Optional

In [26]:
from langchain_classic.memory import InMemoryEntityStore


entity_store = InMemoryEntityStore()
entity_memory = ConversationEntityMemory(
    llm=llm,
    entity_store=entity_store,
    return_messages=True
)

In [28]:
class AgentState(TypedDict):
    messages: List
    entity_context: Optional[str]


def entity_aware_agent_node(state:AgentState):
    user_input = state["messages"][-1].content
    memory_vars = entity_memory.load_memory_variables({"input": user_input})
    entity_context = memory_vars.get("entities", "")
    system_prompt = f"""You are a helpful assistant with memory.
    Use the following known entity facts to inform your response:
    {entity_context}
    """
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_input)
    ])
    entity_memory.save_context(
        {"input": user_input},
        {"output": response.content}
    )
    return {
        "messages": state["messages"] + [response],
        "entity_context": str(entity_context)
    }


checkpointer = MemorySaver()

graph = StateGraph(AgentState)
graph.add_node("agent", entity_aware_agent_node)
graph.set_entry_point("agent")
graph.add_edge("agent", END)

app = graph.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "session-1"}}
app.invoke({
    "messages": [HumanMessage(content="Alice is a software engineer at TechCorp. She loves Python.")],
    "entity_context": None
}, config=config)

app.invoke({
    "messages": [HumanMessage(content="Bob is a data scientist at DataCo. He specializes in NLP.")],
    "entity_context": None
}, config=config)

print(entity_store)

In [ ]:
print(entity_store.store)


In [ ]:
memory_vars = entity_memory.load_memory_variables({"input": "What does Bob specialize in?"})
print(memory_vars)